tech: power bi, sql, python, excel/csv, dax, power query

operations+bi+data analysis(python+sql)

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
print(Path.cwd())

/Volumes/DATA/logist/notebooks


In [2]:
proj=Path.cwd().parent
print(proj)

/Volumes/DATA/logist


In [3]:
DATA_FOLDER = proj / "data"
RAW_DATA = DATA_FOLDER / "raw"
PROCESSED_DATA = DATA_FOLDER / "processed"

RAW_DATA.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA.mkdir(parents=True, exist_ok=True)

print("Project:", proj)
print("Raw data:", RAW_DATA)
print("Processed data:", PROCESSED_DATA)

Project: /Volumes/DATA/logist
Raw data: /Volumes/DATA/logist/data/raw
Processed data: /Volumes/DATA/logist/data/processed


In [197]:
Path.cwd()

PosixPath('/Volumes/MAC/logist/notebooks')

In [198]:
#Dimentional table

warehouses = [
{
"warehouse_id": "WH01",
"warehouse_name": "Brussels Hub",
"city": "Brussels",
"region": "Brussels"
},
{
"warehouse_id": "WH02",
"warehouse_name": "Antwerp Hub",
"city": "Antwerp",
"region": "Flanders"
},
{
"warehouse_id": "WH03",
"warehouse_name": "Liege Hub",
"city": "Liege",
"region": "Wallonia"
},
{
"warehouse_id": "WH04",
"warehouse_name": "Ghent Hub",
"city": "Ghent",
"region": "Flanders"
},
{
"warehouse_id": "WH05",
"warehouse_name": "Charleroi Hub",
"city": "Charleroi",
"region": "Wallonia"
}
]

shifts = [
{
"shift_id": "S1",
"shift_name": "Morning",
"start_time": "06:00",
"end_time": "14:00"
},
{
"shift_id": "S2",
"shift_name": "Afternoon",
"start_time": "14:00",
"end_time": "22:00"
},
{
"shift_id": "S3",
"shift_name": "Night",
"start_time": "22:00",
"end_time": "06:00"
}
]

products = [
{
"product_id": "P001",
"product_category": "Parcel",
"weight_category": "Light"
},
{
"product_id": "P002",
"product_category": "Parcel",
"weight_category": "Medium"
},
{
"product_id": "P003",
"product_category": "Parcel",
"weight_category": "Heavy"
},
{
"product_id": "P004",
"product_category": "Express",
"weight_category": "Light"
},
{
"product_id": "P005",
"product_category": "Express",
"weight_category": "Medium"
}
]

In [199]:
warehouses=pd.DataFrame(warehouses)
warehouses.to_csv(RAW_DATA/"warehouses.csv", index=False)

shifts=pd.DataFrame(shifts)
warehouses.to_csv(RAW_DATA/"shifts.csv", index=False)

products=pd.DataFrame(products)
warehouses.to_csv(RAW_DATA/"products.csv", index=False)

In [200]:
np.random.randint(1000,5000)

2565

In [201]:
dates=pd.date_range(start="2025-01-01", end="2025-12-31", freq="D")

In [202]:
opr = []

for date in dates:
  for warehouse_id in warehouses["warehouse_id"]:
     for shift_id in shifts["shift_id"]:
        opr.append({
        "date": date,
        "warehouse_id": warehouse_id,
        "shift_id": shift_id
})

opr = pd.DataFrame(opr)

print("Number of records:", len(opr))

Number of records: 5475


In [203]:
opr["date"].nunique()

365

In [204]:
opr.head()

,date,warehouse_id,shift_id
0,2025-01-01,WH01,S1
1,2025-01-01,WH01,S2
2,2025-01-01,WH01,S3
3,2025-01-01,WH02,S1
4,2025-01-01,WH02,S2


In [205]:
print("dupl",opr.duplicated().sum())

dupl 0


In [206]:
opr.to_csv(RAW_DATA/"opr_skeleton.csv", index=False)


warehouse_productivity={"WH01": 85,
"WH02": 90,
"WH03": 75,
"WH04": 82,
"WH05": 70}

In [207]:
warehouse_base_volume = {
"WH01": 4200,
"WH02": 5000,
"WH03": 3500,
"WH04": 3800,
"WH05": 3000
}



monthly_multiplier = {
1: 0.85,
2: 0.80,
3: 0.90,
4: 0.95,
5: 1.00,
6: 1.00,
7: 0.90,
8: 0.85,
9: 1.00,
10: 1.05,
11: 1.25,
12: 1.50
}


weekday_multiplier = {
0: 1.05, # Monday
1: 1.00, # Tuesday
2: 1.00, # Wednesday
3: 1.05, # Thursday
4: 1.10, # Friday
5: 0.65, # Saturday
6: 0.55 # Sunday
}

shift_volume_share = {
"S1": 0.40,
"S2": 0.35,
"S3": 0.25
}


opr["month"] = opr["date"].dt.month
opr["weekday"] = opr["date"].dt.weekday

#13.6
opr["base_volume"] = opr["warehouse_id"].map(warehouse_base_volume)

opr.head()

#13.7
opr["monthly_multiplier"] = opr["month"].map(monthly_multiplier)


opr[["date", "warehouse_id", "month", "monthly_multiplier"]].head()


opr["weekday_multiplier"] = opr["weekday"].map(weekday_multiplier)


opr[["date", "weekday", "weekday_multiplier"]].head()


opr["shift_share"] = opr["shift_id"].map(shift_volume_share)


opr[["shift_id", "shift_share"]].drop_duplicates()


opr["expected_volume"] = (opr["base_volume"]* opr["monthly_multiplier"]* opr["weekday_multiplier"]* opr["shift_share"])


opr["expected_volume"] = (opr["expected_volume"].round().astype(int))


random_variation = np.random.normal(loc=1.0,scale=0.10,size=len(opr))


opr["volume"] = (opr["expected_volume"]* random_variation).round().astype(int)


opr[["date","warehouse_id","shift_id","volume"]].head()


opr["shipments"] = opr["volume"]
opr[["date", "warehouse_id", "shift_id", "shipments"]].head()

#13.13--end

,date,warehouse_id,shift_id,month,weekday,base_volume
0,2025-01-01,WH01,S1,1,2,4200
1,2025-01-01,WH01,S2,1,2,4200
2,2025-01-01,WH01,S3,1,2,4200
3,2025-01-01,WH02,S1,1,2,5000
4,2025-01-01,WH02,S2,1,2,5000


,date,warehouse_id,month,monthly_multiplier
0,2025-01-01,WH01,1,0.85
1,2025-01-01,WH01,1,0.85
2,2025-01-01,WH01,1,0.85
3,2025-01-01,WH02,1,0.85
4,2025-01-01,WH02,1,0.85


,date,weekday,weekday_multiplier
0,2025-01-01,2,1.0
1,2025-01-01,2,1.0
2,2025-01-01,2,1.0
3,2025-01-01,2,1.0
4,2025-01-01,2,1.0


,shift_id,shift_share
0,S1,0.40
1,S2,0.35
2,S3,0.25


,date,warehouse_id,shift_id,volume
0,2025-01-01,WH01,S1,1469
1,2025-01-01,WH01,S2,1264
2,2025-01-01,WH01,S3,936
3,2025-01-01,WH02,S1,1756
4,2025-01-01,WH02,S2,1367


,date,warehouse_id,shift_id,shipments
0,2025-01-01,WH01,S1,1469
1,2025-01-01,WH01,S2,1264
2,2025-01-01,WH01,S3,936
3,2025-01-01,WH02,S1,1756
4,2025-01-01,WH02,S2,1367


In [208]:
#13.14
opr.groupby("warehouse_id")["shipments"].mean().round()
opr.groupby("shift_id")["shipments"].mean().round()
opr.groupby("month")["shipments"].mean().round()

warehouse_id
WH01    1287.0
WH02    1536.0
WH03    1074.0
WH04    1164.0
WH05     917.0
Name: shipments, dtype: float64

shift_id
S1    1429.0
S2    1256.0
S3     902.0
Name: shipments, dtype: float64

month
1     1024.0
2      948.0
3     1053.0
4     1136.0
5     1188.0
6     1181.0
7     1081.0
8     1000.0
9     1192.0
10    1267.0
11    1459.0
12    1804.0
Name: shipments, dtype: float64

In [209]:
base_workers = {
"WH01": 25,
"WH02": 30,
"WH03": 20,
"WH04": 23,
"WH05": 18
}

shift_worker_multiplier = {
"S1": 1.00, # Morning
"S2": 0.95, # Afternoon
"S3": 0.75 # Night
}

#14.3
opr["base_workers"] = opr["warehouse_id"].map(base_workers)

opr["shift_worker_multiplier"] = opr["shift_id"].map(shift_worker_multiplier)


opr["workers"] = (opr["base_workers"]* opr["shift_worker_multiplier"]).round().astype(int)


staffing_variation = np.random.normal(loc=1.0,scale=0.05,size=len(opr))


opr[["date", "warehouse_id", "shift_id", "workers"]].head()


,date,warehouse_id,shift_id,workers
0,2025-01-01,WH01,S1,25
1,2025-01-01,WH01,S2,24
2,2025-01-01,WH01,S3,19
3,2025-01-01,WH02,S1,30
4,2025-01-01,WH02,S2,28


In [210]:
#15
SHIFT_HOURS = 8

opr["labor_hours"] = (opr["workers"] * SHIFT_HOURS)

opr[["date","warehouse_id","shift_id","workers","labor_hours","shipments"]].head()




,date,warehouse_id,shift_id,workers,labor_hours,shipments
0,2025-01-01,WH01,S1,25,200,1469
1,2025-01-01,WH01,S2,24,192,1264
2,2025-01-01,WH01,S3,19,152,936
3,2025-01-01,WH02,S1,30,240,1756
4,2025-01-01,WH02,S2,28,224,1367


*PRODUCTIVITY Calc*

In [211]:
#KPIs 17

opr["Productivity"]=opr["shipments"]/ opr["labor_hours"].round(2)
opr[["Productivity","warehouse_id","shift_id","workers","labor_hours","shipments"]].head()


,Productivity,warehouse_id,shift_id,workers,labor_hours,shipments
0,7.345000,WH01,S1,25,200,1469
1,6.583333,WH01,S2,24,192,1264
2,6.157895,WH01,S3,19,152,936
3,7.316667,WH02,S1,30,240,1756
4,6.102679,WH02,S2,28,224,1367


# Business Anlaysis

In [212]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [213]:
#18
#warehouse with highest productivity
opr.groupby("warehouse_id")["Productivity"].mean()

#shift with highest productivity
opr.groupby("shift_id")["Productivity"].mean()

#staffing
#warehouse with highest productivity
opr.groupby("warehouse_id")["Productivity"].mean()

opr.groupby("shift_id")["workers"]


warehouse_id
WH01    7.038374
WH02    7.144453
WH03    7.404425
WH04    6.992939
WH05    6.959531
Name: Productivity, dtype: float64

shift_id
S1    7.706933
S2    7.140478
S3    6.476423
Name: Productivity, dtype: float64

warehouse_id
WH01    7.038374
WH02    7.144453
WH03    7.404425
WH04    6.992939
WH05    6.959531
Name: Productivity, dtype: float64

In [214]:
#20 define processing capacity
base_productivity = {
"WH01": 85,
"WH02": 90,
"WH03": 75,
"WH04": 82,
"WH05": 70
}# base_productivity

base_workers

shift_productivity_multiplier = {
"S1": 1.05, # Morning
"S2": 1.00, # Afternoon
"S3": 0.85 # Night
}#shift_productivity_multiplier


opr["base_productivity"] = opr["warehouse_id"].map(
base_productivity
)#base_productivity

opr["shift_productivity_multiplier"] = opr["shift_id"].map(
shift_productivity_multiplier
)#shift_productivity_multiplier


opr["expected_productivity"] = (
opr["base_productivity"]
* opr["shift_productivity_multiplier"]
)#expected_productivity


#Capacity=hrs labours*expected productivity
SHIFT_HOURS = 8
opr["labor_hours"] = (opr["workers"] * SHIFT_HOURS)

opr["capacity"]=(opr["labor_hours"]*opr["expected_productivity"]).round().astype(int)


opr[
[
"date",
"warehouse_id",
"shift_id",
"workers",
"labor_hours",
"shipments","capacity"
]
].head(20)





{'WH01': 25, 'WH02': 30, 'WH03': 20, 'WH04': 23, 'WH05': 18}

,date,warehouse_id,shift_id,workers,labor_hours,shipments,capacity
0,2025-01-01,WH01,S1,25,200,1469,17850
1,2025-01-01,WH01,S2,24,192,1264,16320
2,2025-01-01,WH01,S3,19,152,936,10982
3,2025-01-01,WH02,S1,30,240,1756,22680
4,2025-01-01,WH02,S2,28,224,1367,20160
5,2025-01-01,WH02,S3,22,176,1246,13464
6,2025-01-01,WH03,S1,20,160,1197,12600
7,2025-01-01,WH03,S2,19,152,1036,11400
8,2025-01-01,WH03,S3,15,120,734,7650
9,2025-01-01,WH04,S1,23,184,1242,15842


In [215]:
opr["shipments"]=(opr["shipments"]*8).astype(int)#volume scale =8 because shipments/capacity=8, shipments<<<capacity 
opr["productivity"]=(opr["shipments"]/opr["labor_hours"]).round(2)
opr["utilization"]=(opr["shipments"]/opr["capacity"])
opr["utilization_perct"]=(opr["utilization"]*100).round(2)
opr[
[
"date",
"warehouse_id",
"shipments","capacity","utilization"
]
].head(20)
#shipments<<capacity(utilisation<65-95) 

,date,warehouse_id,shipments,capacity,utilization
0,2025-01-01,WH01,11752,17850,0.658375
1,2025-01-01,WH01,10112,16320,0.619608
2,2025-01-01,WH01,7488,10982,0.681843
3,2025-01-01,WH02,14048,22680,0.619400
4,2025-01-01,WH02,10936,20160,0.542460
5,2025-01-01,WH02,9968,13464,0.740345
6,2025-01-01,WH03,9576,12600,0.760000
7,2025-01-01,WH03,8288,11400,0.727018
8,2025-01-01,WH03,5872,7650,0.767582
9,2025-01-01,WH04,9936,15842,0.627194


In [216]:
#xplore 28
#avg utilsation by warehouse
opr.groupby("warehouse_id")["utilization_perct"].mean().round(1)
#wrt shift
opr.groupby("shift_id")["utilization_perct"].mean().round(1)
#no.overcapacity recordslen(over_capacity)



warehouse_id
WH01    68.6
WH02    65.9
WH03    81.9
WH04    70.8
WH05    82.3
Name: utilization_perct, dtype: float64

shift_id
S1    73.6
S2    71.7
S3    76.4
Name: utilization_perct, dtype: float64

In [217]:
#overload operation 29

over_capacity=opr[opr["shipments"]>opr["capacity"]]
#overcapacity records
len(over_capacity)

676

In [218]:
#30 backlog creation
opr["backlog"]=(opr["shipments"]-opr["capacity"]).clip(lower=0)

In [219]:
opr[["capacity","backlog","shipments"]].head()
opr["backlog"].describe()
#ware that have max backlogging
opr.groupby("warehouse_id")["backlog"].mean()

,capacity,backlog,shipments
0,17850,0,11752
1,16320,0,10112
2,10982,0,7488
3,22680,0,14048
4,20160,0,10936


count     5475.000000
mean       256.938995
std        923.792776
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max      10000.000000
Name: backlog, dtype: float64

warehouse_id
WH01    179.510502
WH02    158.085845
WH03    401.333333
WH04    200.084932
WH05    345.680365
Name: backlog, dtype: float64

In [220]:
#correction
units_per_shipment=np.random.choice([1,2,3],size=len(opr),p=[0.6,0.3,0.1])
opr["units_procced"]=(opr["shipments"]*units_per_shipment)


In [221]:
len(over_capacity)

676

In [222]:
#avg util wrt warehouse
opr.groupby("warehouse_id")["utilization_perct"].mean()

#avg backlog wrt warehouse
opr.groupby("warehouse_id")["backlog"].mean().round(0)



warehouse_id
WH01    68.618557
WH02    65.854228
WH03    81.886000
WH04    70.775744
WH05    82.312749
Name: utilization_perct, dtype: float64

warehouse_id
WH01    180.0
WH02    158.0
WH03    401.0
WH04    200.0
WH05    346.0
Name: backlog, dtype: float64

In [223]:
#33.1
base_eroor_rate=0.008 #(8%)

#making high utilization increase the risk
opr["utilization_factor"]=np.where(opr["utilization"]>0.85,1.40,1.00) #if util>85% increase error by 40%



In [224]:
error_rate_variation=np.random.normal(loc=1.0,scale=0.15,size=len(opr))
opr["error_rate"]=(error_rate_variation*opr["utilization_factor"]*base_eroor_rate)
opr["error_rate"]=opr["error_rate"].clip(lower=0.001,upper=0.05)#making sure it never becomes negative(0.1-5%)

In [232]:
#no.of errors
opr["errors"]=(opr["shipments"]*opr["error_rate"]).round().astype(int)
opr["errors_percet"]=(opr["errors"]/opr["shipments"]*100).round(2)

In [233]:
opr.groupby(opr["utilization"]>0.85)["errors_percet"].mean().round(2)#(0.8-1.1%)

utilization
False    0.80
True     1.11
Name: errors_percet, dtype: float64

In [236]:
opr.groupby(opr["utilization"]>0.85)["errors"].mean().round(2)

utilization
False     69.28
True     132.48
Name: errors, dtype: float64

In [234]:
#error wrt warehouse

opr.groupby("warehouse_id")["errors_percet"].mean().round(2)
opr.groupby("warehouse_id")["errors"].sum()



warehouse_id
WH01    0.85
WH02    0.85
WH03    0.93
WH04    0.86
WH05    0.95
Name: errors_percet, dtype: float64

warehouse_id
WH01     98741
WH02    116794
WH03     91001
WH04     90266
WH05     79003
Name: errors, dtype: int64

In [228]:
opr.columns

Index(['date', 'warehouse_id', 'shift_id', 'month', 'weekday', 'base_volume',
       'monthly_multiplier', 'weekday_multiplier', 'shift_share',
       'expected_volume', 'volume', 'shipments', 'base_workers',
       'shift_worker_multiplier', 'workers', 'labor_hours', 'Productivity',
       'base_productivity', 'shift_productivity_multiplier',
       'expected_productivity', 'capacity', 'productivity', 'utilization',
       'utilization_perct', 'backlog', 'units_procced', 'utilization_factor',
       'error_rate', 'errors', 'errors_percet'],
      dtype='object')

In [235]:
opr["error_rate"].min()
opr["error_rate"].max()
opr["errors_percet"].max()
opr["errors_percet"].min()

0.0032035342445814355

0.017327941836826635

1.74

0.32

# logistic kpis delivery 

In [237]:
#34.1 does the operational pressure affect service performace?
base_on_time_rate=0.96 #96% operational

#if backlog inc. service performance dec.

opr["backlog_factor"]=np.where(opr["backlog"]>500,0.92,np.where(opr["backlog"]>200,0.96,1.00)) #backlog<200 normal 200-500 getting worse, >500 service presure

In [239]:
#34.3
on_time_variation=np.random.normal(loc=1.0,scale=0.01,size=len(opr))

In [241]:
opr["on_time_rate"]=(base_on_time_rate*opr["backlog_factor"]*on_time_variation)
opr["on_time_rate"]=opr["on_time_rate"].clip(lower=0.8,upper=0.99)#to keep btw 80-99%

In [243]:
#on time shipments

opr["on_time_ships"]=(opr["shipments"]*opr["on_time_rate"]).round().astype(int)

#late shipments
opr["late_shipments"]=(opr["shipments"]-opr["on_time_ships"])

opr[["on_time_ships","late_shipments","shipments"]]

,on_time_ships,late_shipments,shipments
0,11328,424,11752
1,9535,577,10112
2,7111,377,7488
3,13648,400,14048
4,10476,460,10936
...,...,...,...
5470,11949,571,12520
5471,10548,1220,11768
5472,12539,1589,14128
5473,12735,1537,14272


In [245]:
opr["on_time_ships_percet"]=(opr["on_time_rate"]*100).round(2)
opr[["on_time_ships","late_shipments","shipments","on_time_ships_percet"]]

,on_time_ships,late_shipments,shipments,on_time_ships_percet
0,11328,424,11752,96.39
1,9535,577,10112,94.30
2,7111,377,7488,94.96
3,13648,400,14048,97.15
4,10476,460,10936,95.79
...,...,...,...,...
5470,11949,571,12520,95.44
5471,10548,1220,11768,89.63
5472,12539,1589,14128,88.76
5473,12735,1537,14272,89.23


*checking higher backlog effecting service*

In [246]:
opr.groupby(opr["backlog"]>500)["on_time_ships_percet"].mean().round(2)

backlog
False    95.96
True     88.30
Name: on_time_ships_percet, dtype: float64

In [248]:
#which warehouse has beset service performance
opr.groupby("warehouse_id")["on_time_ships_percet"].mean().round(2)

#which warehouse has beset service performance
opr.groupby("warehouse_id")["late_shipments"].mean().round(2)

warehouse_id
WH01    95.54
WH02    95.56
WH03    94.78
WH04    95.41
WH05    94.84
Name: on_time_ships_percet, dtype: float64

warehouse_id
WH01    495.16
WH02    587.41
WH03    499.48
WH04    463.82
WH05    423.36
Name: late_shipments, dtype: float64

* the warehouse having high on time shipments delivered is also having high no.of late shipments and also having high shipments numbers

* analyse the created dataset 

In [250]:
#39
opr.to_csv(RAW_DATA/"operations_complete.csv", index=False)
backup=opr.copy()

In [ ]:
#41
